# Water Quality Prediction & Evolutionary Optimisation
**Date:** April 2025  

‼️Please clone the project from GitHub, or update the following paths:
1. File Data path currently set as "../data/e1_nutrients.csv". 
2. If you want the plot images saved, change all plot savefig paths. Currently, all are commented by default.
 
Github link:https://github.com/mzst-26/water-quality-ml

This notebook covers two parts:
- **Part 1:** Predicting nitrite levels in water samples using machine learning regression models (Linear Regression, Random Forest, Neural Network)
- **Part 2:** Optimising the McCormick function using a Hill Climber and Evolutionary Algorithm

---

## Table of Contents
1. [Libraries & Setup](#1-libraries--setup)
2. [Part 1 – Machine Learning](#1)
   - [1.1 Load Data](#1)
   - [1.2 Exploratory Data Analysis](#2.2)
   - [1.3 Data Preparation](#2.3)
   - [1.4 Regression Models](#2.4)
   - [1.5 Model Tuning](#2.5)
   - [1.6 Final Evaluation](#2.6)
3. [Part 2 - Optimisation](#3)
   - [2.1 Generation of Random Solutions](#3.1)
   - [2.2 Algorithms](#3.2)
   - [2.3 Experimemint & Analysis](#3.3)

   

<a id="1-libraries--setup"></a>
---
## 1. Libraries & Setup

In [ ]:
# core libraries
import numpy as np
import pandas as pd

#for visualization
import matplotlib.pyplot as plt
import seaborn as sns

#for checking multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

#image visualisation
from IPython.display import Image, display

#Regression models
import sklearn.linear_model as linearRegression
import sklearn.ensemble as RandomForestRegressor
import sklearn.neural_network as MLPRegressor

#  Model evaluation
from sklearn.model_selection import learning_curve

#Metrices 
from sklearn.metrics import mean_squared_error, r2_score

# Train-test split
from sklearn.model_selection import train_test_split

# For feature scaling
from sklearn.preprocessing import MinMaxScaler

# to control warnings
import warnings

# for 5-fold cross-validation
from sklearn.model_selection import cross_val_score, KFold

# for r2_square
from sklearn.metrics import r2_score

#for model tuning
from sklearn.model_selection import GridSearchCV

#for 3d ploting
from mpl_toolkits.mplot3d import Axes3D

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

<a id="1"></a>
---
# Part 1 – Machine Learning

### 2.1 Load Data

In [ ]:
#load the data using pandas
data_file = pd.read_csv("../data/e1_nutrients.csv")

Used head(), info() and describe() to explore the data and understand the charactiristic of the data.

In [ ]:
data_file.head()        # first 5 rows
data_file.info()        # column names, data types, missing values
data_file.describe()    # min, max, mean, std for every column

<a id="2.2"></a>
---
### 2.2 Exploratory Data Analysis

In [ ]:
# Visualise feature distributions using KDE plots
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
# Flatten the axes array for easy iteration
for ax, col in zip(axes.flatten(), data_file.columns):
    # Create a KDE plot for each feature
    sns.kdeplot(data_file[col], ax=ax, fill=True, color='steelblue')
    ax.set_title(col)

# Adjust layout and show the plot
plt.suptitle('Feature Distributions (KDE)', fontsize=16)
# plt.savefig('../Analytics/1_raw_data_analytics/kde.png', dpi=150)
plt.tight_layout()
plt.show()

### KDE Distribution Observation
>Previously, I chose a histogram plot to visualise the data, which, compared to KDE, was missing some critical information.
#### The KDE reveals:
1. DEPTH The KDE shows 6 distinct peaks, one for each depth value. This actually visually proves it's discrete rather than continuous, far more clearly than the histogram did.
2. NITRATE+NITRITE has a clear bimodal distribution (two humps — one near 0 and a second around 5-6). This is a significant finding. It suggests two distinct water conditions exist in the dataset, possibly seasonal or driven by depth.
3. SILICATE also shows a bimodal shape with peaks around 0.5 and 2. Again suggests two different water states.
4. PHOSPHATE similarly shows a small secondary peak around 1.5 after the main peak near 0.2.

>The KDE plots reveal that Depth takes exactly 6 discrete values (0, 10, 20, 30, 
40 and 60 metres), visible as 6 distinct peaks, confirming a controlled repeated 
sampling design. NITRITE and AMMONIA are heavily right-skewed with a sharp peak 
near zero and a long tail, indicating most samples have low concentrations, with 
Sometimes, extreme readings which would require outlier handling after being confirmed by other visualisation methods. More interestingly, 
NITRATE+NITRITE, SILICATE and PHOSPHATE all display bimodal distributions, two 
distinct peaks. This suggests the dataset captures two different water conditions, 
possibly caused by the seasonal variation or possible depth-related nutrient stratification. 
These patterns suggest the relationships in this data are non-linear and complex.


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data_file.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
# plt.savefig('../Analytics/1_raw_data_analytics/correlation_heatmap.png', dpi=150)
plt.show()

## Correlation Heatmap Observation
What does the heatmap tell me?
1. Looking at the NITRATE+NITRITE and PHOSPHATE numbers(0.80), we can conclude that their relationship is the strongest. When one side is high, we can assume the other side could also be high.
2. NITRATE+NITRITE and SILICATE (0.69) have a strong relationship, and again, when one increases, you can see the increase on the other as well.
3. SILICATE and PHOSPHATE (0.61) Moderate positive relationship, same pattern.
4. AMMONIA and NITRATE+NITRITE (-0.22) Slight negative relationship, slightly unusual.

### Most important
### NITRITE and everything else (0.03 to 0.31)
Our most important nutrient, which is the target variable (for this prediction model), has a low relationship with everything else.

### What does that mean?
>This means predicting nitrite is genuinely difficult, and linear relationships alone won't capture it well. The weak correlations between NITRITE and all features suggest that nitrite behaves somewhat independently from the other nutrients, which means Linear Regression will likely struggle compared to Random Forest or Neural Network, which can capture non-linear relationships. 
#### *** These are predictions I made before even creating the model, solely based on the correlation heatmap. *** #### 
 


In [ ]:
# Box plots to identify outliers
data_file.plot(kind='box', subplots=True, layout=(3, 4), figsize=(16, 10), sharex=False, sharey=False)
plt.suptitle('Box Plots – Outlier Detection', fontsize=16)
plt.tight_layout()
# plt.savefig('../Analytics/1_raw_data_analytics/boxplots.png', dpi=150)
plt.show()

## Box Plot Observation
What does it tell me?
1. DEPTH No outliers, clean even spread across 0-60. Confirms what we already knew: controlled sampling.
2. NITRITE Massive amount of outliers shown as black dots above the whisker. The box itself is tiny and squashed near 0, meaning that 50% of all the values are extremely low, but there are hundreds of readings that are unusually high. This confirms the heavy right skew and tells me outlier handling is definitely needed here.
3. NITRATE+NITRITE Large spread in the box, no extreme outliers shown. More variation in the normal range than NITRITE.
4. AMMONIA shows simmilar behaviour as NITRITE.The box is small and there is a huge dense cluster of outliers above.That means there are hundereds of AMMONIA readings that go well above the normal range.
5. SILICATE Only a handful of outliers (the small circles), and the box has a reasonable spread. Much cleaner than NITRITE or AMMONIA. However the density of the outliers in the 5.8 area is unclear. 
6. PHOSPHATE A few scattered outliers above the whisker but nothing as extreme as NITRITE or AMMONIA. Relatively clean however again the density of the outliers in the narow section of approximitly 1.3 is not very clear. 

### What does that mean for me?

>NITRITE and AMMONIA have the most severe outlier problems, hundreds of data points sitting well above the normal range. These need to be handled in the data preparation step with the IQR method before training the models, otherwise those extreme values will disproportionately influence how the models learn and they skew the results.




In [ ]:
# Pairwise relationships coloured by depth
data_file['Depth_Category'] = data_file['Depth'].astype(str) + 'm'
# Create pairplot with seaborn
sns.pairplot(data_file, 
             hue='Depth_Category', # colour by depth category
             vars=['NITRITE', 'NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE'], # only plot these features
             diag_kind='kde', # use KDE for diagonal plots
             plot_kws={'alpha': 0.3}) # add some transparency to the scatter points to better visualise density and reduce overplotting
plt.suptitle('Pairwise Relationships Coloured by Depth', y=1.02) # adjust title position
# plt.savefig('../Analytics/1_raw_data_analytics/pairplot.png', dpi=150)
plt.show()

# Drop the helper column after plotting
data_file.drop(columns=['Depth_Category'], inplace=True) 

## DIAGONAL (KDE distributions by depth) and OFF-DIAGONAL SCATTER PLOTS
After some research and viewing different tradeoffs, I decided to visualise the data for the last time using this method to confirm some of the guesses we made with the heatmap and the box plots, and we gathered some important data.
For context, the upper diagonal (triangular) and lower diagonal are exact same but with axes swapped. What we are looking for is only learner patterns with low cluster or reliable upward/downward patterns. 

What is not considered reliable: 
>Fan-shaped, Vertical or Horizontal flat cluster(low change, useless relationship), curved patterns (non-linear such as Nitrate vs Silicate) and random clouds.
### Diagonal KDE findings:

1. SILICATE shows the strongest depth separation, and concentration varies most by depth
2. AMMONIA is nearly identical across all depths. Surface-driven process.
3. NITRATE+NITRITE and PHOSPHATE show partial depth separation; deeper samples trend slightly higher.
4. NITRITE overlaps completely across all depths, meaning depth does not drive nitrite levels

### Scatter plot findings:

5. Depth colour never separates any panel, and nutrients vary independently of depth throughout.
6. NITRATE+NITRITE vs PHOSPHATE shows the clearest upward trend, confirms 0.80 correlation on the heatmap, and the same environmental driver
7. NITRATE+NITRITE vs SILICATE also shows a clear upward trend, confirms a 0.69 correlation on the heatmap.
8. NITRITE forms vertical clusters in every panel, with no relationship to any other nutrient, confirming correlation heatmap findings.
9. AMMONIA vs SILICATE and AMMONIA vs PHOSPHATE show near-zero relationships. Ammonia behaves independently.

The pairwise plot reveals much more critical data. Describing them all would indeed result in some long, boring paragraphs. So here are the two most important findings:

1. The depth colour never cleanly separates any scatter plot; the 6 depth levels are randomly mixed throughout every panel, proving that the depth is not the hidden driver behind the nutrients' relationship and that nutrients actually vary independently of depth.
2. Second, NITRITE behaves fundamentally differently from all other variables; it forms dense vertical clusters in every scatter plot it appears in, showing virtually no relationship with any other nutrient. Every other variable shows at least moderate relationships with each other, particularly NITRATE+NITRITE and PHOSPHATE, which show a clear upward trend confirming their 0.80 correlation.




## EDA Summary

>These findings confirm that nitrite is driven by entirely different environmental processes, explaining why predicting it will be genuinely difficult and providing strong evidence that non-linear models will be required.


<a id="2.3"></a>
---
## 2.3 Data Preparation

### How are we going to prepare the data?
1. Check what we are working with.
2. Handle missing values first
3. Handle Outliers
4. Encode DEPTH as Categorical
5. Check Multicollinearity
6. Split Before Scaling
7. Scale Features
8. Final Check, verify Everything Looks Right

### Data Preparation: 1. Check what we are working with.
Before touching anything I would like to understand the scale of the problem. 

In [ ]:
original_len = len(data_file)
print(f"Total rows: {original_len}") # total number of rows
print(f"Total columns: {data_file.shape[1]}") # total number of columns 
print("\nMissing values:")
print(data_file.isnull().sum()) # check if any of the rows have missing values 
print("\nData types:")
print(data_file.dtypes) #understand the data types
print("\nBasic statistics:") 
print(data_file.describe()) #repeated step just to review the data again before preparation

#### Overview
The dataset is large enough to cap rather than drop. The data is overall clean, and there are no unexpected types nor values. Also, there is no new information since we have already analysed the numbers before.

#### Since there are no missing values, we can go straight to step 3.

### Data Preparation: 3. Handle Outliers
Since we are dealing with the environment and those extreme readings could be environmental pollution, I have decided not to drop the outliers to minimise losing critical data. 
>In fact, the outliers themselves may be the real normal data and the low reading could be decrese of minirals of water by seasonal or occasional events such as Algal blooms. Another reason for not dropping is that if we don't train our models with some outliers, we are basically claiming that those situations do not exist. When the model encounters a high nutrient in the future, it will predict badly.
>Capping keeps the information that something extreme happened while limiting how much it distorts the model.



In [ ]:
# Check how extreme the outliers are before deciding
for col in data_file.columns:
    # Calculate the IQR for the column IQR stands for Interquartile Range, which is a measure of statistical dispersion and is used to identify outliers in a dataset. 
    # It is calculated as the difference between the third quartile (Q3) and the first quartile (Q1) of the data. 
    # The IQR helps to determine the range within which most of the data points lie, 
    # and any data points that fall outside of this range are considered outliers.
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1 
    # Define outlier thresholds lower in this case means  any value that is less than 1.5 times the IQR below the first quartile which are considered outliers.
    lower = Q1 - 1.5 * IQR 
    upper = Q3 + 1.5 * IQR 
    # Outliers are defined as any value that is less than the lower threshold or greater than the upper threshbold. 
    outliers = data_file[(data_file[col] < lower) | (data_file[col] > upper)]
    # Print the number and percentage of outliers for each column
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(data_file)*100:.1f}%)")

### Output evaluation
The output confirms why capping is the right choice here: 
>As previously mentioned, visualising data helps with understanding the overall patterns and characteristics of the data, but we might miss some information, such as density, when viewing overlapping data.
1. SILICATE: 0.4% (11 rows) and PHOSPHATE: 0.2% (6 rows) show low density of outliers out of threshold, confirming plot boxes were not visually misleading.
2. AMMONIA: 12.5% (311 rows), the worst offender. If we were to drop the outliers, we would have lost 12% of the real environmental data.
3. NITRATE+NITRITE: 0% as expected, confirming the plot box.
4. NITRITE: 8.4% (209 rows) significant but not catastrophic, capping is correct.
5. DEPTH: 0% — confirmed, no outliers.

#### Summary
> This output also validates the markdown reasoning perfectly. AMMONIA at 12.5% proves that dropping would have been wrong; losing over 300 real pollution event readings would have made the model blind to extreme conditions.



### Data Preparation: 3.2 Cap the outliers

In [ ]:
# Cap outliers instead of dropping
cols_to_cap = ['NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE', 'NITRATE+NITRITE']
for col in cols_to_cap:
    # Calculate the IQR for the specific columns column
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1
    # Get the lower and upper thresholds for outliers
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap the outliers by replacing values outside the thresholds with the threshold values
    data_file[col] = data_file[col].clip(lower, upper)
    
# print the confirmation that all outliers have been capped and the number of rows preserved
print(f"All outliers capped. Rows preserved: {len(data_file)}")

#### Code evaluation
>NITRATE+NITRITE was included in the capping loop as a precaution despite showing zero outliers in the initial check, ensuring consistency across the pipeline if the dataset is updated or thresholds are adjusted in future.

#### The next step is to prove that the capping worked

In [ ]:
# Verify capping worked
# The same method that was used to get the outliers is now pasted here, only for a linear experience. 
for col in cols_to_cap:
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data_file[(data_file[col] < lower) | (data_file[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers remaining")


### Data Preparation: 3.3 Changes are easier to comprehend when visualised, let's see the change side by side before and after capping

In [ ]:
urls = []
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/2_After_capping/beforeAfterKDE.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/2_After_capping/beforeAfterBoxPlot.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/2_After_capping/beforeAfterCM.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/2_After_capping/beforeAfterPairPlot.png?raw=true")

for url in urls:
    display(Image(url=url))

## Summary of the most important changes we can see
>The outliers were not just noise; they were suppressing the true correlation between other nutrients and NITRITE. Before, hundreds of black dots were above the NITRITE whisker. After, it became completely clean, no dots at all. The Correlation Heatmap shows improvements between the relationship of NITRITE and other nutrients, such as NITRATE, SILICATE and PHOSPHATE. The most dramatic change being the NITRATE vs PHOSPHATE, which doubled from 0.15 to 0.30.

>NITRITE correlations with other variables strengthened significantly after capping, bimodal distributions emerged in NITRITE and AMMONIA, and scatter plot relationships became more visible. This confirms that the outliers were distorting the true signal in the data rather than representing meaningful variation, and that the cleaned dataset will give models a far clearer signal to learn from.

### Data Preparation: 4 Encode DEPTH as Categorical

In [ ]:
# Encode the 'Depth' column using one-hot encoding if it exists and hasn't been encoded yet
# This is done on the memory level, not on disk, so the original data file remains the same.
if 'Depth' in data_file.columns:
    # Created one-hot encoded columns for each unique value in the 'Depth' column, with a prefix of 'Depth' for the new column names
    depth_dummies = pd.get_dummies(data_file['Depth'], prefix='Depth', dtype=int) # dtype=int to get 0/1 instead of True/False
    # Concatenate the original dataframe with the new one-hot encoded columns
    data_file = pd.concat([data_file, depth_dummies], axis=1)
    # Drop the original 'Depth' column as it is now encoded into the new columns
    data_file.drop(columns=['Depth'], inplace=True)
    print("Depth column found and encoded successfully.")
    
elif any(col.startswith('Depth_') for col in data_file.columns):
    print("Depth encoding already exists (Depth_ columns found).")
else:
    print("No Depth column and no Depth_ columns found.")

print(data_file.columns)
# Top 5 rows of the final prepared data file
data_file.head()

#### Why does this step matter?
>Previously, we had Debt as a number (0, 10, 20, 30, 40, 60). The model would read the ddepthas continues and assumes that depth 60 is 6 times more important than 10. But they are only locations and not a factor of whether it's important. That is why converting them to dummy columns tells the model there are 6 categories of depth with no relationship between them.

#### When would I not do this?
>If Depth has a continuous relationship with Nitrite, meaning the deeper it goes, the more/less Nitrite we'd find, then we would keep it as numeric. But the pairwise plots showed that depth colour never separated cleanly from other variables, which indicates the relationship is not smooth or continuous.



### Data Preparation: 5  Check Multicollinearity

In [ ]:
# create a copy of the data file without the target variable 'NITRITE' to check for multicollinearity among the features
temp_copy = data_file.drop(columns=['NITRITE'])
# Create an empty DataFrame to store VIF values
vif_data = pd.DataFrame()
# Fill the first column of the table with the feature names
vif_data['Feature'] = temp_copy.columns
# for each feature calculate the VIF score 
# What it actually does internally is run a regression predicting that feature from all other features and measure how well it can be predicted. 
# High predictability = high VIF = redundant feature.
vif_data['VIF'] = [variance_inflation_factor(temp_copy.values, i) 
                   for i in range(temp_copy.shape[1])]

#Prints the results sorted from highest VIF to lowest so the most problematic features appear first.
print(vif_data.sort_values('VIF', ascending=False))

#### VIF GUID:
- Below 5: low multicollinearity, feature is contributing independently
- 5 to 10: moderate, worth monitoring
- Above 10: severe, feature is largely redundant
#### VIF Results Evaluation
>VIF analisis confirmed that all the features contribute independently with a score of blow 5 meanning no problimatic multicollinearity, despite the correlation observed between the NITRATE+NITRITE and PHOSPHRATE (0.85). Each feature contributes sufficiently unique information. No features were removed as a result of this analysis.




### Data Preparation: 6 Split Before Scaling


In [ ]:
# Copy dataset without NITRITE for input 
X = data_file.drop(columns=['NITRITE'])

# Target variable, which is only the NITRITE column
y = data_file['NITRITE']

# Create four separate variables for the training and testing sets using an 80-20 split of the data.
# X_train is the feature rows the model trains from
# X_test is the feature rows the model is tested on after training
# y_train is the correct nitrite answers for the training rows
# y_test is the correct nitrite answers for the test rows, used to check predictions against
# Before training, the function randomly shuffles all rows so the split is not biased towards any particular order(random_state=42 ensures every run has the same exact split)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # Reserves 20% of rows for testing and uses 80% for training.
    random_state=42     # ensures same split on every run
)

# confirm the split worked by outputting the number of rows in the training and testing sets
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

#### Evaluation of the split
>The slit was successful, as the output confirms there are 1996 rows for training and 500 testing rows. 

### Data Preparation: 7 Scale Features

In [ ]:
# Feature scaling using Min-Max Scaling
# Min-Max Scaling is a technique that transforms features to a common scale without distorting differences in the ranges of values.
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)  # fit AND transform on training data
X_test_scaled = scaler.transform(X_test)         # ONLY transform on test data, never fit


#### Why Scale at all?
>Silicate goes up to 8, Phosphate stays below 2 and Depth after encoding is 0 or 1. If we don't scale them, the neural network treats Sl as more important than Phosphate only because the numbers are bigger. To prevent the model from valuing the scale of the numbers rather than their actual predictive value, scaling is used.

#### Why did I choose MinMax over standard scaling?
>I was going to use the standard scaling, which converts everything to have a mean of 0 and a standard deviation of 1, but this calculation would assume that the numbers are normally distributed. The data is heavily skewed and not normally distributed. The MinMax scuashes everything between 0 and 1 regardless of how distributed the data is. 

#### Why only fit the training data?
>fit_transform on training data calculates the minimum and maximum from training rows only. transform on test data applies those same training minimums and maximums to the test rows. This means the scaler never sees the test data during fitting, preserving the integrity of the evaluation.

### Data Preparation:8 Final Check, Verify Everything Looks Right

In [ ]:
print("Training set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)
print("\nTarget variable stats after preparation:")
print(y_train.describe())
print("\nAnyw nulls in training set:", 
      pd.DataFrame(X_train_scaled).isnull().sum().sum())

#### Summary 
The final prepared dataset contains 1996 training rows and 500 test rows across 10 features. Zero null values remain in the training set. Data preparation is complete and the dataset is ready for modelling.

<a id="2.4"></a>
---
## Regression Models

#### Regression Models that will be used:
- Linear Regression: Linear Regression draws a posible straight line through the data. If we plot Silicate(x) and Nitrite(y), it finds a line that minimises the distance between itself and all the points. Linear Regression is very fast but can only find linear relationships. Since the EDA indicated Nitrite with a more complex relationship we can assume that Linear Regression will strugle finding a relationship.
- Random Forest creates a number of trees, each trained with a random subset of the data and the output is the average output of all. Since each tree trains on a sepreate data, they all make unique mistakes. Therefore averaging the outputs cancel the mistakes out. I have used this method called ensemble learning.
- MLPRegressor is a simple neurol network. Data enters the input layer, then goes through hidden layers, each hidden layer increasingly detects patterns and the output is the prediction. First layers detect simple patterns and as it goes through more layers, more complex patterns are detected. This is most time and power consuming compare to the other models.

### Regression Models: Step one, Initialising Models


In [ ]:
# Linear Regression has no important parameters to set; it simply finds a straight line through the data
models = {
    'linearRegression': linearRegression.LinearRegression(),

    'randomForest': RandomForestRegressor.RandomForestRegressor(
        n_estimators=200,       # created 200 trees to reduce variance 
        max_depth=None,         # trees grow until leaves are pure
        min_samples_split=3,    # minimum samples to split a node to prevent overfitting
        random_state=42
    ),

    'neuralNetwork': MLPRegressor.MLPRegressor(
        hidden_layer_sizes=(256, 128, 64, 32, 16),  # 5-layer pyramid
        activation='relu',          # rectified linear unit to avoid vanishing gradients
        solver='adam',              # Adaptive learning rate optimiser
        learning_rate='adaptive',   # this reduces learning rate when loss plateaus
        max_iter=1000,              # 1000 iterations for a deeper network to converge
        early_stopping=True,        # stops training if validation score stops improving
        validation_fraction=0.1,    # 10% of training data used for early stopping check
        random_state=42
    )
}

print("All models are initialised. Start Training")  #Just so we get an output

> Now that the models are initialised, we can train all three models

### Regression Models: Step two, Training Models

In [ ]:
# We train each model on the training data with .fit() from sklearn
# warnings.filterwarnings('ignore')  # suppress convergence warnings during training
# fit each model to the training data
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"{name} — trained") # just so we have an output

# Check Neural Network convergence separately after the loop
nn = models['neuralNetwork']
print(f"\nNeural Network converged: {nn.n_iter_ < nn.max_iter}")
print(f"Iterations used: {nn.n_iter_} / {nn.max_iter}")

#### What does .fit() function do?
- For Linear Regression, it mathemathicly calculats the best coaficient for each feature. This is done with a formula called Ordinary Least Sequance. 

- For Random Forest it creates a number of trees based on the input of n_estimators. Each of the trees access different part of training data. 

- For MLP Regressor the fit function repeatedly passes all training rows through the network, then calculates how wrong the predictions are, and adjusts the internal weights slightly to reduce that error. Does this max_iter number of times.

### Regression Models: Step three, Sample predictions 

In [ ]:
# Comparison table
# create a dictionary to store the predicitons
results = {}
# for each model, predict the first 5 rows of the test set and round it to 4 decimal points
for name, model in models.items():
    results[name] = model.predict(X_test_scaled[:15]).round(4) 

comparison = pd.DataFrame(results) # create a dataframe from the prediction dictionary.
comparison.insert(0, 'Actual', y_test[:15].values) # insert the actual nitrite values from the target test set. 
# output the comparison table.
print("Sample Predictions — All Models vs Actual:") 
print(comparison.to_string(index=False))

#### Evaluation
The 0.515 rows (rows 5, 12, 14) are consistently under-predicted by all models, which could indicate a pattern of struggling with larger values. 
0.0000 values are handled very well by the Random Forest model.

Random forest did best overall, predicting low nitrite levels very well. Did not predicted higher value of Nitrite well. Closest on mid-range values generally.

The neural network is decent on mid-range values, noticeably over-predicts some low values (e.g. 0.050 actual, 0.2089 predicted. That is a huge miss). Struggles with the extremes in both directions

Linear Regression is never close to zero on the zero rows (always 0.03+), and consistently the furthest from actual on most rows but does okay on mid-range.

The problem with 0.515 could be from a lack of enough examples with that level of nitrite, or the features don't have a strong enough signal at that range.

### Assessment of Regression

#### Cross_Validaiton

In [ ]:
# 5-fold cross-validation setup
# KFold is a cross-validator that divides the dataset into k consecutive folds (without shuffling by default).
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cross_validator_results = {}
# for every model, we run 5-fold cross-validation
for name, model in models.items():
    # cross_val_score runs the model training, then returns the evaluation scores for each fold.
    scores = cross_val_score(
        model, X_train_scaled, y_train, # the model and the training data to evaluate on
        cv=kf,                          # the strategy for splitting the data into folds
        scoring='neg_mean_squared_error' # the metric to evaluate, negative MSE because sklearn expects higher is better, but MSE is lower is better
    )
    mse_scores = -scores  # convert back to positive MSE.
    cross_validator_results[name] = mse_scores # store the MSE scores for each fold

    # print the results for each model
    print(f"{name}")
    print(f"  Fold MSEs : {mse_scores.round(6)}")
    print(f"  Mean MSE  : {mse_scores.mean():.6f}")
    print(f"  Std MSE   : {mse_scores.std():.6f}\n")



### Why cross-validation?

Judging on only one train/test split gives a misleading MSE score. The split can be in favour or disadvantage a specific model by chance.

5-fold cross-validation splits the training data into 5 equal parts. 
The model is trained on 4 parts and tested on the remaining 1, repeated 
5 times, so every row gets tested exactly once. This gives 5 MSE scores 
per model. This is a more reliable picture of true performance than a 
single split.

I used neg_mean_squared_error because cross_val_score always treats bigger scores as better.
Since lower MSE is better, sklearn flips its sign during scoring.
After scoring, I flip it back so I can report normal (positive) MSE values.



#### Visualise the cross validation using plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Define consistent colors for each model across both plots
colors = {
    'linearRegression': 'steelblue',
    'randomForest':     'mediumpurple',
    'neuralNetwork':    'coral'
}
# define the fold numbers for the x-axis of the left plot
folds = [1, 2, 3, 4, 5]


# Left plot — MSE per fold
for name, scores in cross_validator_results.items():
    # plot the MSE scores for each fold with markers and lines, using the right colors
    ax1.plot(folds, scores, marker='o', label=name,
             color=colors[name], linewidth=1.5)


# Set labels, title, legend, and grid for the left plot
ax1.set_xlabel('Fold')
ax1.set_ylabel('MSE')
ax1.set_title('MSE per Cross-Validation Fold')
ax1.legend()
ax1.grid(True, alpha=0.3)


# Right plot — Mean MSE with std error bars
names  = list(cross_validator_results.keys())
means  = [cross_validator_results[n].mean() for n in names] # calculate the mean MSE for each model to plot as bars
stds   = [cross_validator_results[n].std()  for n in names] # calculate the standard deviation of MSE for each model to plot as error bars


bars   = ax2.bar(names, means, color=[colors[n] for n in names], # use the same colors as the left plot for consistency
                 alpha=0.8, edgecolor='white') 
ax2.errorbar(names, means, yerr=stds, fmt='none',
             color='black', capsize=5, linewidth=1.5) # add error bars to show the standard deviation of MSE across folds for each model
ax2.set_ylabel('Mean MSE') # label for the y-axis of the right plot

ax2.set_title('Mean Cross-Validation MSE (± std)') # the ± std is just to clarify what the error bars represent
ax2.tick_params(axis='x', rotation=10) # rotate x-axis labels for better readability
ax2.grid(True, alpha=0.3, axis='y') # add horizontal grid lines to the right plot for easier comparison of mean MSE values

plt.tight_layout()
# plt.savefig('../Analytics/3_cross_validator/cross_validator_results.png', dpi=150, bbox_inches='tight') 
plt.show()

### Evaluation of 5-fold cross-validation
> Random Forest is the strongest model, it achieved the lowest mean squared errors, out of all 5 folds. Linear regresson once more confirms the finding of the EDA. It is struggling to find any linear relationship and has the worst predictions out of all three models. Neural Network is in between, but showes better results, possibly it could do better with either a larger dataset or a bit of tuning.

#### Random Forest statistics:
>randomForest RMSE = √0.006026 = 0.078218 μmol/L
<br></br>
>√0.006026 /  0.210601 * 100 = 36.85%
>This means, our best performing model is on average off by 36.85% from the actual prediction. Even though Random Forest is the best performing model 37% is a big error. 

- This could be due to the size of the dataset and/or indicates that there isn't enough information for the model to find a better relationship. 
- Another factor that could be contributing towards this big error could be the way I have prepared the data. NITRITE max is 2.9 but 75% of values are below 0.23, the extreme values could be distorting the model. 

<a id="2.5"></a>
---
## 2.5. Model Tuning

#### 2.5 - 1 Log-transform

Why?

We will be using GridSearchCV to find the best model parameters, but if our data is skewed, GridSearchCV will find the best parameters for a skewed problem, not for the actual data. 

In [ ]:

# Apply log(1 + y) transformation to reduce skewness
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
                      
# Compare distributions before and after
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot original distribution
ax1.hist(y_train, bins=50, color='coral', alpha=0.8)
ax1.set_title('NITRITE — Original Distribution')
ax1.set_xlabel('NITRITE (μmol/L)')
ax1.set_ylabel('Frequency')

# Plot the log-transformed distribution on the right
ax2.hist(y_train_log, bins=50, color='steelblue', alpha=0.8)
ax2.set_title('NITRITE — Log Transformed Distribution')
ax2.set_xlabel('log(1 + NITRITE)')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 2.5 - 2. GridSearchCV on Random Forest


In [ ]:
# initials parameter grid for Rnadom Forest, which I will use for hyperparameter tuning with GridSearchCV.
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4]
}

# GridSearchCV is a method to systematically work through multiple combinations of parameter tunes, 
# cross-validating as it goes to determine which tune gives the best performance.
grid_search = GridSearchCV(
    RandomForestRegressor.RandomForestRegressor(random_state=42),
    param_grid,
    cv=5, # 5-fold cross-validation to evaluate each combination of parameters
    scoring='neg_mean_squared_error', 
    n_jobs=-1, 
    verbose=1
)

# Fit the GridSearchCV object to the training data to find the best hyperparameters based on cross-validated MSE.
grid_search.fit(X_train_scaled, y_train_log)
 

# After fitting, we can access the best parameters and the corresponding MSE score.
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best MSE        : {-grid_search.best_score_:.6f}")
print(f"Best RMSE       : {np.sqrt(-grid_search.best_score_):.6f}")


# the best_estimator_ attribute gives us the Random Forest model trained with the best hyperparameters found by GridSearchCV, ready to be evaluated on the test set or used for predictions.
best_rf = grid_search.best_estimator_

#### Evaluation fo results
GridSearchCV tested 3^4 = 81 different combinations of parameters. Each combination was tested with 5-fold cross-validation, so it trained and evaluated a Random Forest 405 times in total to find the winner.
Best mean squared error of 0.003746 
Best RMSE 0.061204

Achieved with the pratamers of:
max_depth = None meaning each tree is allowed to grow as deep as it needs to. Is seems like limiting depth was not helpful with this dataset. 
min_samples_leaf = 1 meaning each leaf node representing one datapoint, again prefered no restriciton.
min_samples_split = 2  meaning a node only needs 2 samples to be worth splitting. Minimal restriction.
n_estimators = 300 meaning 300 trees outperformed 100 and 200. More trees = more reliable averaging of predictions.

### 2.5 -3 Tuning Neural Network manually

In [ ]:
# Test different architectures systematically
architectures = {
    'Small  (64, 32)':           (64, 32), # smaller network with 2 layers to reduce overfitting risk
    'Medium (128, 64, 32)':      (128, 64, 32),    # a common architecture for tabular data, more capacity than the small one
    'Large  (256,128,64,32,16)': (256, 128, 64, 32, 16) # largest network with 5 layers for maximum capacity   (This is what I tried in the initial tranining)
}

nn_results = {} # dictionary to store the MSE results for each architecture
# loop through each architecture and evaluate it using the 5-fold cross-validation on the training data.
for arch_name, arch in architectures.items():
    nn_test = MLPRegressor.MLPRegressor(
        hidden_layer_sizes=arch, # use the current architecture from the loop
        activation='relu', # rectified linear unit to avoid vanishing gradients
        solver='adam', # Adaptive learning rate optimiser
        learning_rate='adaptive', # this reduces learning rate when loss plateaus.
        max_iter=1000, # 1000 iterations for a deeper network to have enough time to converge
        early_stopping=True, # stops training if validation score stops improving, to prevent overfitting.
        validation_fraction=0.1, # 10% of training data used for early stopping check
        random_state=42
    )
    
    scores = cross_val_score(
        nn_test, X_train_scaled, y_train_log, # the model and the training data to evaluate on
        cv=5, #5 folds
        scoring='neg_mean_squared_error' 
    )
    # convert the negative MSE scores back to positive and calculate MSE
    mse = -scores.mean()
    # store the mean MSE for the current architecture in the results dictionary
    nn_results[arch_name] = mse
    # print the MSE and RMSE for the current architecture.
    print(f"{arch_name:30s} Mean MSE: {mse:.6f}  RMSE: {np.sqrt(mse):.6f}")

# After evaluating all architectures, we can find the one with the lowest MSE by using the min function on the results dictionary.
best_arch_name = min(nn_results, key=lambda k: float(nn_results[k]))

print(f"\nBest architecture: {best_arch_name}")

#### Result evaluation
> Best combination determined by the cross validation is:
Small (64, 32)
With a Mean MSE of 0.007389


### 2.5 - 4 Before and After Tuning Comparison

In [ ]:

print("Final Model Comparison — All results in original scale")

# We wont tune linear regression since it has no hyperparameters. 
print("""
      
Linear Regression — NOT TUNED
  No hyperparameters to optimise.
  RMSE (final) : 0.139754 
      
""")

# random forest tuning results
# Convert best_rf predictions back to the original scale
# get the predictions from the best_rf model on the test set.
rf_predictions_original = np.expm1(best_rf.predict(X_test_scaled))
# convert the log_transformed test target back to the original scale with the inverse of the log1p.
y_test_original= np.expm1(y_test_log)
# calculate the RMSE on the original scale by comparing the predictions from the best_rf model with the original target values.
rf_tuned_rmse = np.sqrt(((rf_predictions_original - y_test_original) ** 2).mean())


# print
print("Random Forest — TUNED with GridSearchCV + log transform")

print(f"  Before tuning : RMSE = 0.077628  (CV on original y_train)")
print(f"  After tuning  : RMSE = {rf_tuned_rmse:.6f} (best_rf on original scale)")
print(f"  Improvement   : {((0.077628 - rf_tuned_rmse) / 0.077628) * 100:+.1f}%") # calculate the persentage improvement from tuning
print(f"  Best params   : {grid_search.best_params_}") # print the best hyperparameters found by GridSearchCV for the Random Forest model


# neural network tuning results
# Retrain best architecture on log target then convert back
best_nn_final = MLPRegressor.MLPRegressor(
    hidden_layer_sizes=(64, 32),   # Small won the architecture search
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)
# fit the best model architecture with the log-transformed target variables.
best_nn_final.fit(X_train_scaled, y_train_log)

# get the predicitons from the original best_nn_final model on the test set, which are currently in the log-transformed scale, 
nn_predictions_original = np.expm1(best_nn_final.predict(X_test_scaled))
# convert the log_transformed test target back to the original scale with the inverse of the log1p.
nn_tuned_rmse = np.sqrt(((nn_predictions_original - y_test_original) ** 2).mean())


print(f"""
      
Neural Network — TUNED via architecture search + log transform
      
  Before tuning : RMSE = 0.107853  (CV on original y_train)
  After tuning  : RMSE = {nn_tuned_rmse:.6f}  (best architecture on original scale)
  Improvement   : {((0.107853 - nn_tuned_rmse) / 0.107853) * 100:+.1f}% 
  Best architecture : Small (64, 32) — larger architectures overfit this dataset
""")

# calculate the final ranking

print("-" * 50)
print("Final Model Ranking (lowest root mean square error is the best)")

final_results = {
    'Linear Regression': 0.139754,
    'Random Forest':     rf_tuned_rmse,
    'Neural Network':    nn_tuned_rmse
}

# sort the final results by RMSE and print the ranking
for i, (name, rmse) in enumerate(sorted(final_results.items(), key=lambda x: x[1]), 1):
    print(f"  {i}. {name:20s} RMSE = {rmse:.6f}")

#### Approach Evaluation
The comparison could not be done straightforwardly. Why?
 - The original cross-validation was `y-train`, the original scale, but the tuning CV was run on `y_train_log`, log scale.

 They produce values in different units, so they can't be compared directly. Instead, we can compare them fairly once we convert them all to RMSE μmol/L. 


### Visualise comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Define consistent colors for each model across both plots
models_names = ['Linear Regression', 'Random Forest', 'Neural Network']
colors = ['steelblue', 'mediumpurple', 'coral']

# Data for the bar charts
before_rmse = [0.139754, 0.077628, 0.107853]
after_rmse  = [0.139754, rf_tuned_rmse, nn_tuned_rmse]

# Set up the x-axis positions for the grouped bar chart
x = np.arange(len(models_names))
width = 0.35 # width of the bars in the grouped bar chart

# left plot, grouped bar chart comparing before and after RMSE for each model
bars1 = ax1.bar(x - width/2, before_rmse, width, label='Before tuning', color=colors, alpha=0.4, edgecolor='black', linewidth=0.8)
bars2 = ax1.bar(x + width/2, after_rmse, width, label='After tuning', color=colors, alpha=0.9,edgecolor='black', linewidth=0.8)

# Add value labels on top of each bar
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2,
    bar.get_height() + 0.002,
    f'{bar.get_height():.4f}',
    ha='center', va='bottom', fontsize=8)
# the same for the after tuning bars, which will show the improvement clearly when compared to the before tuning bars.
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2,
    bar.get_height() + 0.002,
    f'{bar.get_height():.4f}',
    ha='center', va='bottom', fontsize=8)

# Set labels, title, legend, and grid for the left plot
ax1.set_ylabel('RMSE (μmol/L)')
ax1.set_title('Before vs After Tuning — RMSE Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(models_names, rotation=10)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Add annotation for Linear Regression
ax1.annotate('Not tuned —\nno change',
             xy=(0, 0.139754),
             xytext=(0.3, 0.15),
             fontsize=8,
             color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

# Right plot for improvement percentage
improvements = [
    0,  # Linear Regression — not tuned
    ((0.077628 - rf_tuned_rmse) / 0.077628) * 100, # calculate the prescentage improvement for the Random Forest model from tuning
    ((0.107853 - nn_tuned_rmse) / 0.107853) * 100
]

bar_colors = ['lightgray', 'mediumpurple', 'coral']
bars3 = ax2.bar(models_names, improvements,
                color=bar_colors, alpha=0.85,
                edgecolor='black', linewidth=0.8)

# Add value labels
for bar, imp in zip(bars3, improvements):
    label = f'{imp:.1f}%' if imp > 0 else 'Not tuned' # Show "Not tuned" for the linear regression bar
    ax2.text(
             bar.get_x() + bar.get_width()/2, 
             bar.get_height() + 0.3,   
             label,
             ha='center', va='bottom', fontsize=9,
             fontweight='bold')

ax2.set_ylabel('RMSE Improvement (%)')
ax2.set_title('Improvement Gained from Tuning')
ax2.tick_params(axis='x', rotation=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0, color='black', linewidth=0.8)

# Add annotation for Linear Regression
ax2.annotate('Not tuned —\nno change',
              xy=(0, 0),
              xytext=(0.3, 0.5),
              fontsize=8,
              color='gray',
              arrowprops=dict(arrowstyle='->', color='gray'))

plt.suptitle('Model Tuning Evaluation — Before and After',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Analytics/3_cross_validator/tuning_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

<a id="2.6"></a>
---
## 2.6 Final Evaluation

The Random Forest model achieved the best performance with the lowest MSE, while Linear Regression performed worst and the Neural Network gave moderate results. This is because Random Forest can capture non-linear relationships, whereas Linear Regression assumes linearity and underfits the data. The Neural Network is also capable of modelling complexity but is limited by dataset size and sensitivity to parameter choices.

Data preprocessing improved performance, particularly the log transformation which reduced skewness, and feature scaling which was important for the Neural Network. Cross-validation showed that Random Forest had more consistent performance across folds, while the Neural Network showed slightly higher variability.

Hyperparameter tuning improved the Random Forest the most, with smaller gains for the Neural Network and minimal impact on Linear Regression. In terms of bias–variance, Linear Regression has high bias, Random Forest achieves a better balance, and the Neural Network has higher variance.

A key limitation is the dataset size, which may restrict model performance, especially for the Neural Network. In practice, Random Forest would be the preferred model due to its accuracy and stability, although Linear Regression may still be useful for interpretability.

<a id="3"></a>
---
# 3. Optimisation
In this section, we will explore two optimisation algorithms, Hill Climber and an Evolutionary Algorithm, applied to minimise the McCormick function. Then we will compare the two to measure their performance. The McCormick function is a two-variable mathematical function (𝑓(𝑥, 𝑦) = sin(𝑥 + 𝑦) + (𝑥 − 𝑦)^2 − 1.5𝑥 + 2.5𝑦 + 1)` that has a complex landscape(lots of high and low points making it seem like there is a local solution). This means it has multiple local optima that can trap simple search algorithms before they find the true global minimum. 

> The global minimum is`f(x, y) ≈ −5.0548` at the point of `(x = -3.6888, y = -4.6888)`.
I got the true global minimum within the [-5, 5] search space I used scipy's minimize function with 1000 random starting points to actually know weather the amgorithms performing well or not.  

## Table of contents
 - Task 2.1 — Generation of Random Solutions
 - Task 2.2 — Algorithm Implementation
 - Task 2.3 — Analysis of Results
<a id="3.1"></a>
### Step one: Generation of Random Solutions


In [ ]:
# The McCormick function. Returns the value of the function for the given x and y inputs. 
def mccormick(x, y):
    #𝑓(𝑥, 𝑦)= sin(𝑥 + 𝑦)  + (𝑥 − 𝑦)^2  − 1.5𝑥  + 2.5𝑦  + 1 side by side
    return np.sin(x + y) + ((x - y)**2) - (1.5*x) + (2.5*y) + 1
# Generate 500 random solutions for x and y within the range of -5 to 5, and calculate their fitness using the McCormick function.
np.random.seed(42)
solutions = np.random.uniform(-5, 5, size=(500, 2))
fitness = mccormick(solutions[:, 0], solutions[:, 1])

fig = plt.figure(figsize=(16, 7))

# 2D scatter
ax1 = fig.add_subplot(121)
sc = ax1.scatter(solutions[:, 0], solutions[:, 1], c=fitness, cmap='viridis')
plt.colorbar(sc, ax=ax1, label='Fitness f(x,y)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('500 Random Solutions - 2D')

# 3D surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('f(x, y)')
ax2.set_title('McCormick Surface - 3D View')
ax2.view_init(elev=30, azim=225)

plt.tight_layout()
plt.show()

Here I generated 500 random solutions by sampling x and y from a uniform distribution between -5 and 5. Each point is coloured by its fitness value, darker colours represent lower (better) fitness. You can see from the plot that the best solutions cluster towards the lower-left area of the search space, which is where the known global minimum sits at point of (x = -3.6888, y = -4.6888). The spread of colours in both view shows the function has a complex landscape, meaning random search alone is unlikely to reliably find the global minimum optimum.

<a id="3.2"></a>
### Optimisation - Algorithms


In [ ]:
#  the Gaussian mutation function.
def gaussian_mutation(solution, sigma=0.9):
    mutated = solution + np.random.normal(0, sigma, size=solution.shape) # add Gaussian noise with mean 0 and standard deviation sigma to each element of the solution
    return np.clip(mutated, -5, 5) # ensure 

# the hillclimber function.
def hillclimber(n_iterations=1000, sigma=0.9, seed=None):
    # if seed is provided set the random seed for reproductibility. So that every time it's run with the same seed. 
    if seed is not None:
        np.random.seed(seed) # set the random seed

    # initialise the random solution with a bound of -5 to 5 for both x and y.
    parent = np.random.uniform(-5, 5, size=2)
    # calculate the fitness of the initial solution using the McCormick function.
    parent_fitness = mccormick(*parent)
    # create an archive to store the fitness values of the best solution at each iteration, starting with the initial fitness.
    fitness_archive = [parent_fitness]

    
    for _ in range(n_iterations):
        child = gaussian_mutation(parent, sigma) # set the child 
        child_fitness = mccormick(*child) # calculate the fitness level of the child

        # if the child has better fitness (lower in this case since we are minimising), replace the parent with the child and update the parent fitness.
        if child_fitness < parent_fitness:
            parent = child
            parent_fitness = child_fitness
        # append the current best fitness to the archive after each iteration, 
        # regardless of whether it was updated or not, to track the progress over time.
        fitness_archive.append(parent_fitness)

    return parent, parent_fitness, fitness_archive

# Evelutionary algorithm funciton
def evolutionary_algorithm(pop_size=30, n_iterations=1000, sigma=0.9, seed=None):
    if seed is not None: # if we have seed, set it for reproducibility.
        np.random.seed(seed)
    # Initialise a population of random solutions for x and y within the range of -5 and 5 
    # and calculate their fitness using the McCormick function.
    population = np.random.uniform(-5, 5, size=(pop_size, 2))
    pop_fitness = np.array([mccormick(*ind) for ind in population])
    fitness_archive = [pop_fitness.min()]

    # for every iteration
    for _ in range(n_iterations):
        # create a new generation of children by applying Gaussian mutation to each individual in the current population, and calculate their fitness.
        children = np.array([gaussian_mutation(ind, sigma) for ind in population])
        child_fitness = np.array([mccormick(*ind) for ind in children])

        # Combine the current population and the children, 
        # and select the best individuals based on fitness to form the next generation.
        combined = np.vstack([population, children])
        combined_fitness = np.concatenate([pop_fitness, child_fitness])
        best_idx = np.argsort(combined_fitness)[:pop_size]
        # Update the population and fitness for the next iteration with the best 
        # individuals from the combined set of parents and children.
        population = combined[best_idx]
        pop_fitness = combined_fitness[best_idx]

        # append the best fitness.
        fitness_archive.append(pop_fitness.min())
    # After all iterations, return the best solution found, its fitness, and the archive of fitness values over time.
    best_idx = np.argmin(pop_fitness)
    return population[best_idx], pop_fitness[best_idx], fitness_archive

print("Function definition completed. Start optimization.")


#### Description of algorithems
I have implemented two algorithmes:
 - Hill climber: Hill climber starts with a single random solution and on each iteration it generates one child. That is done using Gaussian mutation. If the child has a better, meaning lower fitness level, then it's replaced with the parent, otherwise it is kept. Hill climber is simple and fast but it struggles with complex landscapes. 

- Evollutionary Algorithm: This works similar to the hill climber, however instead of dealing with a single point at a time, it maintains a poluations of 30 solutions at once. For each iteration, every individual is mutated to produce a child, then the best 30 cobination of child and parents are kept for the next iteration. Having multipul sotions running and exploring at the same time, prevents the algorithm from getting stuck. 

Both algorithms use Gaussian mutation which adds random noise drawn from a normal distribution to each variable. The sigma value (set to 0.3) controls how large these changes are, a bigger sigma explores more broadly, a smaller one makes finer adjustments.



### Experimemint & Analysis
<a id="3.3"></a>

In [ ]:
N_REPEATS = 30 # number of times to repeat each algorithm to get an average performance
N_ITERATIONS = 1000 # number of iterations for each run of the algorithms, which is how long they will seach for better solutions.

hc_archives = [] # store the fitness archives for each run of the hill climber.
ea_archives = [] # store the fitness archives for each run of the evolutionary algorithm.

# Run both algorithms N_REPEATS times.
for i in range(N_REPEATS):
    _, _, hc_arch = hillclimber(n_iterations=N_ITERATIONS, seed=i) # specific seed for reproducibility, so each run is the same every time we run the code.
    _, _, ea_arch = evolutionary_algorithm(n_iterations=N_ITERATIONS, seed=i) 
    # append the fitness archives for track performance. 
    hc_archives.append(hc_arch) 
    ea_archives.append(ea_arch)

# convert the list of archives to numpy arrays. )
hc_archives = np.array(hc_archives) 
ea_archives = np.array(ea_archives)
# print the average.
print(f"Hill Climber  - mean final fitness: {hc_archives[:, -1].mean():.4f}")
print(f"Evol. Alg.    - mean final fitness: {ea_archives[:, -1].mean():.4f}")
print(f"Global minimum: −5.0548")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence plot
axes[0].plot(iterations, hc_mean, label='Hill climber', color='steelblue')
axes[0].plot(iterations, ea_mean, label='Evolutionary algorithm', color='darkorange')
axes[0].axhline(-5.0548, color='green', linestyle='--', linewidth=1, label='Global minimum (-5.0548)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Best fitness')
axes[0].set_title('Convergence Comparison - Mean of 30 Runs')
axes[0].legend()

# Box plot
axes[1].boxplot([hc_archives[:, -1], ea_archives[:, -1]],
                tick_labels=['Hill Climber', 'Evolutionary Algorithm'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.6))
axes[1].axhline(-5.0548, color='green', linestyle='--', linewidth=1, label='Global minimum (-5.0548)')
axes[1].set_ylabel('Final fitness')
axes[1].set_title('Distribution of Final Fitness - 30 Runs')
axes[1].legend()

plt.tight_layout()
# plt.savefig("../Analytics/4_HC_vs_EA/HC_vs_EA.png")
plt.show()


I ran both algorithms 30 times. Each of them had a different random seed to get a reliable picture of how they perform on average, since both use randomness and a single run could be misleading.

> The convergence plot shows the evolutionary algorithm reaches the global minimum of -5.0548 within the first 50 iterations and stays there for all 1000 iterations. The hill climber improves quickly at first but stalls around -3.48 on average, well above the global minimum. 

#### How consistent are they across runs?
 The box plot shows this perfectly; the EA is essentially a single dot at -5.0548. This means all 30 runs found the same answer, which is the global minimum. The hill climber box is very wide, stretching from around -2 down to -5, showing its performance is highly inconsistent and heavily dependent on where it randomly initialises. This means the hill climber is not reliable for this problem. We can not trust a single run to give a good answer. 

#### What role does sigma play?
Initially, the sigma for all of them was 0.1 and 0.3, meaning I did not allow room for exploration. The best result of EA was -4.83. When increasing the sigma to 0.9 for the highest exploration, EA performed much better, increasing to -5.0547.
The mutation step size sigma is set to 0.9 for both algorithms. This controls how large each mutation step is. A larger sigma allows bigger jumps across the search space and as I mentioned earlier, this helps the algorithms explore broadly rather than just refining solutions locally. With a smaller sigma, such as 0.1, both algorithms converged too quickly and got stuck, showing that the choice of sigma has a significant impact on performance.

#### Why the algorithms behave differently?
The fundamental difference is that the Evolutionary Algorithm with 30 solutions spread across the search space is less likely to have all the individuals trapped in the same local optimum. At least some individuals will be in better regions and will pull the population towards the global minimum. On the other side, the hill climber with only tracking one local optimum at a time has no such a safety net. If it starts near a local optimum, it would converge there and never leave.


#### Overall conclutions
> For the McCormick problem between Hill climber and Evolutionary Algorithm, the EA is a much better approch. It consistently finds the global minimum across all 30 runs, while the hill climber averages only -3.48, meaning it get stuck on local optimum. Of course, the trade of here is the comutational costs. The EA runs 30 fitness evaluations per iteration which is much larger compare to just 1 for the hill climber, making it 30 times more expensive per iteration. However for a problem like this where fitness evaluations are cheap, we can negligt the cost and the improvement in solution quality is more valuable.

 
